
# Module 10 — Session 2: Integrating Multiple Messy Sources

This notebook demonstrates a practical integration pipeline:
- Schema alignment and datatype harmonization
- Source precedence and conflict resolution
- Provenance tracking
- Integrating semi-structured JSON
- Synthetic identifiers
- Post-merge duplicate detection
- Validation after integration
- Performance notes


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6,4)
pd.set_option("display.max_columns", 50)



## 1) Create synthetic sources


In [ ]:

import pandas as pd
import numpy as np

erp = pd.DataFrame({
    "customer_id": [1001,1002,1003,1004,1005,1006],
    "name": ["John Smith","Jane Doe","Emily Davis","Michael Johnson","Sarah Scott","Robert Brown"],
    "email": ["john@acme.com","jane@acme.com","emily@acme.com","mike@acme.com","sarah@acme.com","rob@acme.com"],
    "city": ["Knoxville","Nashville","Knoxville","Memphis","Chattanooga","Nashville"],
    "zip": ["37996","37201","37996","38103","37402","37201"],
    "updated_at": pd.to_datetime(["2025-10-01","2025-10-02","2025-10-03","2025-10-04","2025-10-05","2025-10-06"])
})

crm = pd.DataFrame({
    "CustomerNumber": [1001,1002,1003,1004,1007],
    "full_name": ["John Smith","Jane Doe","Emily Davis","Michael Johnson","Karen Adams"],
    "email": ["john.smith@company.com", None, "emily@acme.com","m.johnson@foo.com","karen@bar.com"],
    "City": ["Knoxville","Nashville","Knoxville","Memphis","Nashville"],
    "PostalCode": ["37996","37201","37996","38103","37201"],
    "updated_at": pd.to_datetime(["2025-09-28","2025-09-30","2025-10-03","2025-10-01","2025-10-07"])
})

web = pd.DataFrame({
    "cust_id": [1001,1003,1008],
    "Name": ["John A. Smith","Emily D. Davis","Pat Lewis"],
    "email": ["john@acme.com","emily_davis@example.org","plewis@misc.com"],
    "city": ["Knoxville","Knoxville","Nashville"],
    "zip_code": ["37996","37996","37201"],
    "updated_at": pd.to_datetime(["2025-09-20","2025-10-03","2025-10-03"])
})

print("ERP")
display(erp)
print("\nCRM")
display(crm)
print("\nWeb")
display(web)



## 2) Schema alignment


In [ ]:
def clean_cols(df):
    df = df.copy()
    df.columns = (df.columns
                    .str.strip()
                    .str.lower()
                    .str.replace(' ', '_', regex=False))
    return df

erp_std = clean_cols(erp)
crm_std = clean_cols(crm).rename(columns={
    "customernumber": "customer_id",
    "full_name": "name",
    "city": "city",
    "postalcode": "zip"
})
web_std = clean_cols(web).rename(columns={
    "cust_id": "customer_id",
    "name": "name",
    "zip_code": "zip"
})

for df in (erp_std, crm_std, web_std):
    df["customer_id"] = df["customer_id"].astype(int)
    df["zip"] = df["zip"].astype(str)

print("ERP")
display(erp_std.head())
print("\nCRM")
display(crm_std.head())
print("\nWeb")
display(web_std.head())


## 3) Reference schema validation


In [ ]:

expected_cols = {"customer_id","name","email","city","zip","updated_at"}

def validate_schema(df, name):
    missing = expected_cols - set(df.columns)
    extra = set(df.columns) - expected_cols
    print(f"[{name}] missing={missing} extra={extra} rows={len(df)}")

validate_schema(erp_std, "ERP")
validate_schema(crm_std, "CRM")
validate_schema(web_std, "Web")


## 4) Merge with source precedence


In [ ]:

m = (erp_std
     .merge(crm_std, on="customer_id", how="outer", suffixes=("_erp","_crm"))
     .merge(web_std, on="customer_id", how="outer", suffixes=("","_web"))
    )

def coalesce_3(a,b,c):
    return a if pd.notna(a) else (b if pd.notna(b) else c)

final = pd.DataFrame({
    "customer_id": m["customer_id"],
    "name":  [coalesce_3(a,b,c) for a,b,c in zip(m["name_erp"],  m["name_crm"],  m["name"])],
    "email": [coalesce_3(a,b,c) for a,b,c in zip(m["email_erp"], m["email_crm"], m["email"])],
    "city":  [coalesce_3(a,b,c) for a,b,c in zip(m["city_erp"],  m["city_crm"],  m["city"])],
    "zip":   [coalesce_3(a,b,c) for a,b,c in zip(m["zip_erp"],   m["zip_crm"],   m["zip"])],
})

final.head(10)


## 5) Provenance tracking and conflict resolution by recency


In [ ]:

# Source flags
final["name_source"] = np.select(
    [m["name_erp"].notna(),  m["name_crm"].notna(),  m["name"].notna()],
    ["ERP",                   "CRM",                 "WEB"],
    default="NA"
)

final["email_source"] = np.select(
    [m["email_erp"].notna(), m["email_crm"].notna(), m["email"].notna()],
    ["ERP",                   "CRM",                  "WEB"],
    default="NA"
)


# Recency-based tie-break for email
def pick_recent(row, cols, updated_cols):
    vals  = [row.get(c) for c in cols]
    times = [row.get(u) for u in updated_cols]
    pairs = [(v,t) for v,t in zip(vals, times) if pd.notna(v) and pd.notna(t)]
    return sorted(pairs, key=lambda x: x[1], reverse=True)[0][0] if pairs else np.nan

m["email_recent"] = m.apply(
    lambda r: pick_recent(r,
                          ["email_erp","email_crm","email"],
                          ["updated_at_erp","updated_at_crm","updated_at"]),
    axis=1
)

final.head(10)


## 6) Integrate semi-structured JSON


In [ ]:

from pandas import json_normalize

api_payload = [
    {"customer_id": 1001, "profile": {"name": "John Smith"}, "contacts": {"email": "john@new.com"}, "meta": {"ts": "2025-10-08"}},
    {"customer_id": 1009, "profile": {"name": "Laura Rogers"}, "contacts": {"email": "l.rogers@example.com"}, "meta": {"ts": "2025-10-09"}}
]
df_api = json_normalize(api_payload).rename(columns={"profile.name":"name","contacts.email":"email"})[["customer_id","name","email"]]
df_api["city"] = np.nan
df_api["zip"] = np.nan
df_api["name_source"] = "API"
df_api["email_source"] = "API"

df_all = pd.concat([final, df_api], ignore_index=True)
df_all.sort_values("customer_id").head(12)



## 7) Identifiers: UUID and synthetic hash


In [ ]:

import uuid, hashlib

df_all["uuid"] = [uuid.uuid4().hex for _ in range(len(df_all))]

def synth_id(row):
    key = f"{str(row.get('name')).lower()}|{str(row.get('city')).lower()}|{str(row.get('zip')).lower()}"
    return hashlib.md5(key.encode()).hexdigest()

df_all["synthetic_id"] = df_all.apply(synth_id, axis=1)
df_all.head(10)



## 8) Detect remaining duplicates (simple hybrid rule)


In [ ]:
def char_sim(a, b):
    a, b = (a or ""), (b or "")
    n, m = len(a), len(b)
    if n == 0 and m == 0: return 1.0
    dp = [list(range(m+1))] + [[i] + [0]*m for i in range(1, n+1)]
    for i in range(1, n+1):
        for j in range(1, m+1):
            dp[i][j] = min(
                dp[i-1][j] + 1, dp[i][j-1] + 1,
                dp[i-1][j-1] + (a[i-1].lower() != b[j-1].lower())
            )
    dist = dp[n][m]
    return 1 - dist / max(n, m)

def possible_duplicate(r1, r2, thr=0.8):
    same_email = pd.notna(r1["email"]) and pd.notna(r2["email"]) and r1["email"].lower()==r2["email"].lower()
    same_loc = (r1["city"]==r2["city"]) or (r1["zip"]==r2["zip"])
    similar_name = char_sim(str(r1["name"]), str(r2["name"])) >= thr
    return same_email or (same_loc and similar_name)

pairs = []
df = df_all.reset_index(drop=True)
for i in range(len(df)):
    for j in range(i+1, len(df)):
        if possible_duplicate(df.loc[i], df.loc[j]):
            pairs.append((i,j))

len(pairs), pairs[:5]


## 9) Validation after integration


In [ ]:

# Unique key check where available
assert final["customer_id"].is_unique

# Completeness on final
completeness = final.notna().mean()
print("Field completeness (final):")
print(completeness.sort_values())

# Null rates
null_rates = final.isna().mean().sort_values(ascending=False)
print("\nNull rates (final):")
print(null_rates)



## 10) Performance notes


In [ ]:
print("Chunking example: for chunk in pd.read_csv('big.csv', chunksize=100_000): process(chunk)")
final.info(memory_usage='deep')
print("%timeit final.merge(final, on='customer_id', how='inner')  # example benchmark")


## 11) Summary

- Align schemas and types before merging
- Apply precedence and recency rules for conflicts
- Track provenance
- Re-run duplicate checks after integration
- Validate key properties before publishing the master table
